In [ ]:
#!pip install sentence_transformers
#!pip install ag2[retrievechat]
#!pip install pyautogen==0.7.2
#https://microsoft.github.io/autogen/0.2/docs/notebooks/agentchat_RetrieveChat/

Setting base for Retrive Chat

In [1]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS

config_list = autogen.config_list_from_json("OAI_CONFIG_LIST.json")

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


models to use:  ['gpt-35-turbo', 'gpt-4-vision-preview', 'dalle', 'gemini-pro', 'gemini-1.5-pro-001', 'gemini-1.5-pro', 'gemini-pro-vision']


In [2]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [3]:
config_list = [
    {
        # Let's choose the Meta's Llama 3.1 model (model names must match Ollama exactly)
        "model": "llama3",
        # We specify the API Type as 'ollama' so it uses the Ollama client class
        "api_type": "ollama",
        "stream": False,
        #"client_host": "http://192.168.0.1:11434",
    }
]

In [4]:
import logging
logging.raiseExceptions = False

import agentops
agentops.init("f9de56a1-8b24-408e-a137-7e9638f47479")  
print("AgentOps initialized successfully")

🖇 AgentOps: [ConcurrentFuturesInstrumentor] Successfully instrumented concurrent.futures.ThreadPoolExecutor


AgentOps initialized successfully


🖇 AgentOps: You're on the agentops free plan 🤔


In [5]:
# 1. create an AssistantAgent instance named "assistant"
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)


In [6]:
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Extract text from local PDF files
pdf_files = ["book.pdf"]
#docs_content = [extract_text_from_pdf(file) for file in pdf_files]

from autogen.retrieve_utils import TEXT_FORMATS,extract_text_from_pdf
docs_content= extract_text_from_pdf("book.pdf")

In [7]:
# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["autogen-docs/book.pdf"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback¶

In [8]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "summarize the content and use python code to any calculation given in the book"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)
#agentops.end_session("Success")

Trying to create collection.


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
max_tokens is too small to fit a single line of text. Breaking this line:
	The 22 ...
Failed to split docs with must_break_at_empty_line being True, set to False.
2026-11-16 15:53:48,213 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 210 chunks.
2026-11-16 15:53:48,233 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Model llama3 not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['c024e487', 'e6980caf', '330fa96a', 'ae7d3b76', 'a0e7c3ac', '7b45c013', '363de674', '5385d2ca', '0acb4ca9', 'a3e8f8c3', 'ff103f25', '4a59535b', 'a1dcc20c', '8b0b85f0', '797e9c8f', 'd2307215', 'bfa56a62', '687a8b1d', '94c25c16', '194591db']]
Adding content of doc c024e487 to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc e6980caf to context.


Model llama3 not found. Using cl100k_base encoding.


... (29 outputs left out)

Model llama3 not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: summarize the content and use python code to any calculation given in the book

Context is: 79
THE LAW OF SACRIFICE
22 Laws of Marketing  10/31/02  12:23 PM  Page 79the company looked at the books and decided to focus
on the only product it made money on: toys. As long as
Interstate was going to focus on toys, it decided to
change its name to Toys “R” Us. Today Toys “R” Us
does 20 percent of the retail toy business in the coun-
try. Very profitably, too. In its last fiscal year, Toys “R”
Us made $326 million on sales of $5.5 billion.
Many retail chains are successfully patterning
themselves on the Toys “R” Us formula: a narrow
focus with in-depth sto

🖇 AgentOps: Error in span llama3.llm: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_REPACK buffer of size 3925868544
error loading model: unable to allocate CPU_REPACK buffer (status code: 500)
🖇 AgentOps: Error in span ag2.agent.assistant.receive: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_REPACK buffer of size 3925868544
error loading model: unable to allocate CPU_REPACK buffer (status code: 500)
🖇 AgentOps: Error in span ag2.chat.ragproxyagent_to_assistant.workflow: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_

ResponseError: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_REPACK buffer of size 3925868544
error loading model: unable to allocate CPU_REPACK buffer (status code: 500)

Process Starts for Scenario 2 : Answer a question based off docstrings w/o human feedback

In [9]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "You are a creative head . do many brainstorms"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)

Model llama3 not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['9f39e066', '17300681', 'cea6b5ea', '7a179f8f', '5bb144e0', 'bbe2135b', '413c9d39', 'd48e9565', '7173c3da', '122e5901', '326cee40', 'b3f77776', 'de729254', '50525f55', 'd5c9a510', 'a1dcc20c', 'dba71e1a', 'f8240566', '3c4a80ea', '4245c1d1']]
Adding content of doc 9f39e066 to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc 17300681 to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc cea6b5ea to context.


... (26 outputs left out)

Model llama3 not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: You are a creative head . do many brainstorms

Context is: Without adequate funding
an idea won’t get off the ground.
124
22 Laws of Marketing  10/31/02  12:23 PM  Page 124If you have a good idea and you’ve picked up this
book with the thought in mind that all you need is a lit-
tle marketing help, this chapter will throw cold water
on that thought.
Even the best idea in the world won’t go very far
without the money to get it off the ground. Inventors,
entrepreneurs, and assorted idea generators seem to
think that all their good ideas need is professional mar-
keting help.
Nothing could be further from the truth. Marketing
is a game fought in the min

🖇 AgentOps: Error in span llama3.llm: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_REPACK buffer of size 3925868544
error loading model: unable to allocate CPU_REPACK buffer (status code: 500)
🖇 AgentOps: Error in span ag2.agent.assistant.receive: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_REPACK buffer of size 3925868544
error loading model: unable to allocate CPU_REPACK buffer (status code: 500)
🖇 AgentOps: Error in span ag2.chat.ragproxyagent_to_assistant.workflow: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_

ResponseError: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_REPACK buffer of size 3925868544
error loading model: unable to allocate CPU_REPACK buffer (status code: 500)

Process Starts for Scenario 3 : Generate code based off docstrings w/ human feedback

In [10]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
code_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=code_problem)

Model llama3 not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['a1dcc20c', '660c4d76', 'f8240566', '9f39e066', '7173c3da', 'a0f75e72', 'd15472f5', '73eecb4d', '7a179f8f', 'b0d70839', '1da5c235', 'bbe2135b', '413c9d39', 'be80b4fb', '4e6b17f7', '17300681', '69fa91d4', '55f04e11', '4b5df861', '7b45c013']]
Adding content of doc a1dcc20c to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc 660c4d76 to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc f8240566 to context.


... (28 outputs left out)

Model llama3 not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py

Context is: liant marketing program only to have one of the
immutable laws knock you flat if you don’t know what
they are.
Perhaps it’s human nature not to admit there are
things you can’t do. Certainly most marketers believe
that anything is achievable if you are energetic enough,
or creative enough, or determined enough. Especially
if you are willing to spend enough money.
Once you open your mind to the possibility that
there are laws of marketing, it’s easy to see what they
are. In truth, they are obv

🖇 AgentOps: Error in span llama3.llm: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_REPACK buffer of size 3925868544
error loading model: unable to allocate CPU_REPACK buffer (status code: 500)
🖇 AgentOps: Error in span ag2.agent.assistant.receive: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_REPACK buffer of size 3925868544
error loading model: unable to allocate CPU_REPACK buffer (status code: 500)
🖇 AgentOps: Error in span ag2.chat.ragproxyagent_to_assistant.workflow: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_

ResponseError: llama-server process has terminated: exit status 1: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 3925868544
ggml_backend_buft_alloc_buffer_n_default: failed to allocate CPU_REPACK buffer of size 3925868544
error loading model: unable to allocate CPU_REPACK buffer (status code: 500)